# SC-GRC Hybrid — Final Answer Stage 2/4 — Questions 85–168

This notebook is a **single-run fixed shard**. It loads the completed router and cached SLM representative answers, then generates FT-LLM+RAG answers only for questions that the learned router escalates. Non-escalated questions reuse the already-generated SLM representative answer. No SLM regeneration occurs. No Base/FT/RAG rerun occurs.

Run once, save `/kaggle/working/sc_grc_state` as a Kaggle Dataset, then use it as the input to the next final-answer stage.

In [1]:
%%capture
import os, subprocess, sys
cuda_lib = "/usr/local/cuda/lib64"
ld = os.environ.get("LD_LIBRARY_PATH", "")
if cuda_lib not in ld:
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{ld}"
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",
    "torch==2.3.1","torchvision==0.18.1","--extra-index-url","https://download.pytorch.org/whl/cu121"])
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",
    "pandas==2.2.2","scipy==1.13.1","matplotlib==3.8.4","triton==2.3.1","bitsandbytes==0.44.1",
    "transformers==4.46.3","peft==0.13.2","accelerate==0.34.2","sentencepiece","sacrebleu",
    "rapidfuzz","openpyxl","sentence-transformers","rank_bm25","langchain-text-splitters","faiss-gpu-cu12","FlagEmbedding",
    "scikit-learn","tqdm","bert-score","rouge-score","joblib"])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 330.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 251.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 354.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 286.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 349.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 210.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 327.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 337.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 331.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 320.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 296.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 364.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 233.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 253.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 309.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 326.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 296.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 233.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 300.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 303.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 380.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 381.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 320.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 306.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
pointpats 2.5.5 requires matplotlib>=3.9, but you have matplotlib 3.8.4 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.


In [2]:
# ========================= USER CONFIG =========================
MODEL_ID_SLM = "Qwen/Qwen2.5-3B-Instruct"
SLM_ADAPTER_DIR = "/kaggle/input/datasets/mrnotalent/ewu-qwen-adapter-checkpoint1632"  # EDIT if needed
KB_DIR = "/kaggle/input/datasets/mohuaakter/ewu-dataset-jsonl-pairs/KB/KB"  # KB ONLY; no scraped data
GROUND_TRUTH_XLSX = "/kaggle/input/datasets/mohuaakter/ewu-dataset-jsonl-pairs/University_Chatbot_Questions_1110_GROUND_TRUTHS_VERIFIED.xlsx"  # EDIT
INPUT_STATE_DIR = "/kaggle/input/datasets/bxgdhdgsg/f22222/sc_grc_state"  # EDIT: this shard chains from Notebook 05's saved state (05 -> 06 -> 07 -> 08), NOT directly from Notebook 04 -- it must already contain sc_grc_final_answers.jsonl rows 0:84
WORK_STATE_DIR = "/kaggle/working/sc_grc_state"

SEED = 42
TOP_K_DENSE, TOP_K_SPARSE, TOP_K_FINAL = 10, 10, 4
K_SAMPLES = 3
SC_GRC_TEMPERATURE = 0.7
SC_GRC_TOP_P = 0.9
QUALITY_FLOOR_BERTSCORE = 0.85
MAX_NEW_TOKENS = 200

MODEL_ID_LLM="Qwen/Qwen2.5-7B-Instruct"
LLM_ADAPTER_DIR="/kaggle/input/datasets/mrnotalent/7bbbbb/ewu7b/adapter"  # EDIT: required -- path to your fine-tuned Qwen2.5-7B-Instruct LoRA adapter, loaded below
SHARD_START=84
SHARD_END=168


In [3]:

import os, json, re, shutil, time
from collections import Counter
import numpy as np, pandas as pd, torch, faiss, joblib
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
import sacrebleu

os.makedirs(WORK_STATE_DIR,exist_ok=True)
if INPUT_STATE_DIR and os.path.isdir(INPUT_STATE_DIR) and INPUT_STATE_DIR!=WORK_STATE_DIR:
    for name in os.listdir(INPUT_STATE_DIR):
        src,dst=os.path.join(INPUT_STATE_DIR,name),os.path.join(WORK_STATE_DIR,name)
        if os.path.isfile(src) and not os.path.exists(dst): shutil.copy2(src,dst)

eval_df=pd.read_csv(os.path.join(WORK_STATE_DIR,'eval_333.csv')).sort_values('index' if 'index' in pd.read_csv(os.path.join(WORK_STATE_DIR,'eval_333.csv'),nrows=1).columns else 'Q#') if False else pd.read_csv(os.path.join(WORK_STATE_DIR,'eval_333.csv'))
sig=pd.read_csv(os.path.join(WORK_STATE_DIR,'sc_grc_signals_with_router.csv')).sort_values('idx').reset_index(drop=True)
if len(sig)!=333: raise RuntimeError('Need completed 333-row router state from Notebook 04')
test_references_eval=eval_df['ground_truth'].tolist()

kb_chunks=json.load(open(os.path.join(WORK_STATE_DIR,'kb.json'),encoding='utf-8'))
kb_sources=json.load(open(os.path.join(WORK_STATE_DIR,'kb_meta.json'),encoding='utf-8'))['sources']
dense_index=faiss.read_index(os.path.join(WORK_STATE_DIR,'dense_index.faiss'))
bm25=BM25Okapi([re.findall(r"[\w\u0980-\u09FF]+",c.lower()) for c in kb_chunks])
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
embedder=SentenceTransformer('BAAI/bge-m3',device=DEVICE); embedder.max_seq_length=512
reranker=FlagReranker('BAAI/bge-reranker-v2-m3',use_fp16=True,device=DEVICE)
def embed_texts(texts): return np.asarray(embedder.encode(texts,batch_size=16,normalize_embeddings=False,show_progress_bar=False),dtype='float32')
def hybrid_retrieve(query):
    q = embed_texts([query])
    faiss.normalize_L2(q)

    _, di = dense_index.search(q, min(TOP_K_DENSE, len(kb_chunks)))

    ss = bm25.get_scores(re.findall(r"[\w\u0980-\u09FF]+", query.lower()))
    si = np.argsort(ss)[::-1][:min(TOP_K_SPARSE, len(kb_chunks))]

    ids = sorted(set(di[0].tolist()) | set(si.tolist()))
    pairs = [[query, kb_chunks[i]] for i in ids]

    raw_scores = reranker.compute_score(pairs, normalize=True)

    # Flatten to a 1-D scalar array (same fix as Notebook 04 -- raw output
    # shape/type from FlagReranker.compute_score varies with candidate count).
    rs = np.asarray(raw_scores, dtype=np.float32).reshape(-1)

    if len(rs) != len(ids):
        raise RuntimeError(
            f"Reranker returned {len(rs)} scores for {len(ids)} candidate chunks. "
            f"Raw shape={np.asarray(raw_scores).shape}"
        )

    order = np.argsort(rs)[::-1][:TOP_K_FINAL]

    return [
        {
            'source': kb_sources[ids[int(j)]],
            'chunk': kb_chunks[ids[int(j)]],
            'score': float(rs[int(j)])
        }
        for j in order
    ]

SYSTEM_PROMPT=("You are the East West University (EWU) student support assistant. Answer the student's question directly, accurately, and helpfully. Do not invent university policies or facts. If the provided context is insufficient, say so clearly.")
def build_rag_prompt(query,retrieved):
    ctx='\n\n'.join(f"[{r['source']}]\n{r['chunk']}" for r in retrieved) if retrieved else '(no relevant context retrieved)'
    user=f'Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer using only the context above, following the system rules.'
    return f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n'

# Only the larger FT-LLM is loaded here; SLM samples are already checkpointed.
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16)
tok=AutoTokenizer.from_pretrained(LLM_ADAPTER_DIR,trust_remote_code=True,padding_side='right'); tok.pad_token=tok.pad_token or tok.eos_token
base=AutoModelForCausalLM.from_pretrained(MODEL_ID_LLM,quantization_config=bnb,device_map='auto',trust_remote_code=True,torch_dtype=torch.float16,attn_implementation='sdpa')
llm=PeftModel.from_pretrained(base,LLM_ADAPTER_DIR).eval(); llm.config.use_cache=True
@torch.no_grad()
def generate_llm(query,retrieved):
    p = build_rag_prompt(query, retrieved)

    # Truncate the RAG context, same VRAM-safety guard used in Notebook 04's
    # SLM generation -- the 7B FT-LLM is larger, so this matters even more here.
    e = tok(
        p,
        return_tensors='pt',
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    out = llm.generate(
        **e,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )

    answer = tok.decode(out[0][e['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    del out
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return answer


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [4]:

START=84; END=168
OUT=os.path.join(WORK_STATE_DIR,'sc_grc_final_answers.jsonl')
answers={}
if os.path.exists(OUT):
    with open(OUT,encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r=json.loads(line); answers[int(r['idx'])]=r
if any(i in answers for i in range(START,END)):
    raise RuntimeError(f"Existing answers detected in shard {START}:{END}. Do not rerun this notebook.")
print(f"Processing fixed final-answer shard {START}:{END} = {END-START} questions. Run this notebook once.")
for idx in tqdm(range(START,END),desc=f'SC-GRC final answers {START}:{END}'):
    row=sig.iloc[idx]; q=row['query']; t0=time.time(); escalated=bool(int(row['escalate_pred']))
    if escalated:
        retrieved=hybrid_retrieve(q); answer=generate_llm(q,retrieved); tier='llm'
    else:
        answer=row['representative_answer']; tier='slm'
    r={'idx':idx,'query':q,'answer':answer,'tier':tier,'escalated':escalated,'router_probability':float(row['escalate_proba']),'latency_sec':time.time()-t0}
    answers[idx]=r
    with open(OUT,'a',encoding='utf-8') as f: f.write(json.dumps(r,ensure_ascii=False)+'\n')

if END==333:
    if len(answers)!=333: raise RuntimeError(f'Expected 333 final answers; found {len(answers)}')
    ordered=[answers[i] for i in range(333)]; preds=[r['answer'] for r in ordered]
    rouge=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=False); r1=[];r2=[];rl=[]
    for ref,p in zip(test_references_eval,preds):
        s=rouge.score(ref,p); r1.append(s['rouge1'].fmeasure); r2.append(s['rouge2'].fmeasure); rl.append(s['rougeL'].fmeasure)
    _,_,bf=bert_score_fn(preds,test_references_eval,model_type='bert-base-multilingual-cased',device=DEVICE,batch_size=8,verbose=False)
    chrf=sacrebleu.CHRF(word_order=2); cf=[chrf.sentence_score(p,[r]).score/100 for p,r in zip(preds,test_references_eval)]
    metrics={'N':333,'ROUGE-1':float(np.mean(r1)),'ROUGE-2':float(np.mean(r2)),'ROUGE-L':float(np.mean(rl)),'BERTScore_F1':float(np.mean(bf.tolist())),'chrF++':float(np.mean(cf)),'Escalation_Rate_pct':100*float(np.mean([r['escalated'] for r in ordered])),'Avg_Total_Latency_s':float(np.mean([r['latency_sec'] for r in ordered]))}
    json.dump(metrics,open(os.path.join(WORK_STATE_DIR,'sc_grc_final_metrics.json'),'w'),indent=2)
    pd.DataFrame(ordered).to_csv(os.path.join(WORK_STATE_DIR,'sc_grc_final_answers.csv'),index=False,encoding='utf-8-sig')
    print('\nFINAL SC-GRC METRICS'); [print(k,':',v) for k,v in metrics.items()]; print('Tier counts:',dict(Counter(r['tier'] for r in ordered)) )
else:
    print(f"Shard complete {START}:{END}. Save WORK_STATE_DIR as Dataset and mount it for the next notebook.")


Processing fixed final-answer shard 84:168 = 84 questions. Run this notebook once.


SC-GRC final answers 84:168:   0%|          | 0/84 [00:00<?, ?it/s]


initial target device: 100%|██████████| 2/2 [00:22<00:00, 11.16s/it]

Chunks:   0%|          | 0/2 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

Chunks:  50%|█████     | 1/2 [00:02<00:02,  2.50s/it]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

Chunks: 100%|██████████| 2/2 [00:03<00:00,  1.70s/it]
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/us

Shard complete 84:168. Save WORK_STATE_DIR as Dataset and mount it for the next notebook.
